# 01. Stage 1 counterfactual forecasts and nested selection

The fixed ensemble competes with four guarded individual specifications. On three paired
historical outer windows, compare the fixed ensemble, pooled selection, a pooled default
with a local override, and per-CBSA selection. Every inner fold forecasts 54 months and
scores leads 9–54, with its evaluation completed by the outer forecast origin. A local
override requires at least a 2% inner RMSE improvement and a win on two of the three folds.
The pooling unit is all eligible CBSAs and all observations receive equal region weight.
The production fit uses January 2010–June 2019 and a 54-month forecast.

A challenger replaces the fixed benchmark only if it lowers mean leads 9–54 RMSE at
every outer origin and does not increase the 90th percentile of placebo C46 at any origin.
The overlapping outer windows support a design choice, not an independent final score
for the selected policy. All checks and published CBSA assignments are written to
`output/stage1_nested_selection/`. This step requires the historical panel from 00.


In [13]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

c46_start = Path(os.environ.get("RL_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
c46_root = next((p for p in [c46_start, *c46_start.parents]
                 if (p / "data/processed/omb2018_geography_manifest.json").is_file()), None)
if c46_root is None:
    raise FileNotFoundError("Open the practicum project or set RL_PROJECT_ROOT.")
c46_code = Path(os.environ["RL_C46_CODE"]).resolve() if "RL_C46_CODE" in os.environ else c46_root / "code"
if str(c46_code) not in sys.path:
    sys.path.insert(0, str(c46_code))
c46_output = os.environ.get("RL_C46_OUTPUT")

from c46_forecast_validity import FIXED_ENSEMBLE, LEGACY_FILE, guarded_paths, run_stage1
from stage1_primary import MODEL_FILE, finish_manifest, load_primary_inputs, require_primary
import matplotlib.pyplot as plt
import numpy as np

c46_dir = run_stage1(c46_root, c46_output)
panel, training, forecast, selected = load_primary_inputs(c46_root, c46_dir)
PROC = c46_root / "data/processed"
OUT = c46_root / "output"
forecast.to_csv(c46_root / LEGACY_FILE, index=False)
selected.to_csv(c46_root / MODEL_FILE, index=False)

forecast['date']=pd.to_datetime(forecast.date)
forecast_ids=set(forecast.cbsa_code)
future=panel.loc[panel.cbsa_code.isin(forecast_ids) &
                 panel.date.ge(pd.Timestamp('2020-03-01')),
                 ['cbsa_code','date','employment','cbsa_type']]
gaps=forecast.loc[forecast.date.ge(pd.Timestamp('2020-03-01'))].merge(
    future,on=['cbsa_code','date'],how='left',validate='one_to_one')
if gaps.employment.isna().any() or gaps.counterfactual_employment.isna().any():
    raise ValueError('Missing observed or predicted pandemic-period employment')
if not gaps.groupby('cbsa_code').size().eq(46).all():
    raise ValueError('Expected 46 months (Mar 2020–Dec 2023) for each CBSA')
gaps['gap_pct']=100*(gaps.employment/gaps.counterfactual_employment-1)
gaps['shortfall_pct']=(-gaps.gap_pct).clip(lower=0)
for alternative in ('ensemble','damped','common_ets','short_window'):
    col=f'counterfactual_{alternative}_employment'
    gaps[f'gap_pct_{alternative}']=100*(gaps.employment/gaps[col]-1)
    gaps[f'shortfall_pct_{alternative}']=(-gaps[f'gap_pct_{alternative}']).clip(lower=0)
GAPS_FILE=PROC/'stage1_omb2018_observed_vs_counterfactual_2020_2023.csv'
gaps.to_csv(GAPS_FILE,index=False)

cal=forecast.loc[forecast.date.le(pd.Timestamp('2020-02-01'))].merge(
    panel.loc[panel.cbsa_code.isin(forecast_ids),['cbsa_code','date','employment']],
    on=['cbsa_code','date'],how='left',validate='one_to_one')
if len(cal)!=len(training)*8 or cal.employment.isna().any():
    raise ValueError('Expected eight pre-pandemic diagnostic months per CBSA')
cal['gap_pct']=100*(cal.employment/cal.counterfactual_employment-1)
cal.to_csv(OUT/'stage1_omb2018_precovid_forecast_diagnostic.csv',index=False)

agg=gaps.groupby('date',as_index=False)[['employment','counterfactual_employment']].sum()
fig,ax=plt.subplots(figsize=(10,5))
ax.plot(agg.date,agg.employment,label='Observed covered employment')
ax.plot(agg.date,agg.counterfactual_employment,'--',label='No-pandemic extrapolation')
ax.set(title='2018 OMB CBSAs: observed employment vs counterfactual',ylabel='Covered employment')
ax.legend();fig.tight_layout()
fig.savefig(OUT/'stage1_omb2018_observed_vs_forecast.png',dpi=200)
plt.close(fig)
print('Saved Stage 1 observed-vs-counterfactual:',len(gaps),'rows')
# Pre-pandemic forecast diagnostic using July 2019 through February 2020.
cal['absolute_pct_error']=cal.gap_pct.abs()
cal['pred_minus_actual_pct']=100*(cal.counterfactual_employment/cal.employment-1)
precheck=cal.groupby('cbsa_code').agg(
    mean_gap_pct=('gap_pct','mean'),mean_absolute_pct_gap=('absolute_pct_error','mean'),
    mean_pred_minus_actual_pct=('pred_minus_actual_pct','mean')).reset_index()
precheck.to_csv(OUT/'stage1_omb2018_precovid_diagnostic_by_cbsa.csv',index=False)
print('Pre-pandemic diagnostic, July 2019--February 2020 (not used to choose models):')
print('Region-average gap bias (pp):',round(float(precheck.mean_gap_pct.mean()),3))
print('Region-average absolute gap (pp):',round(float(precheck.mean_absolute_pct_gap.mean()),3))
print('Median region-average absolute gap (pp):',round(float(precheck.mean_absolute_pct_gap.median()),3))
print('Largest diagnostic errors:')
print(precheck.nlargest(10,'mean_absolute_pct_gap').to_string(index=False))
# Note: all figures are diagnostics using observations not eligible for forecasting.

STRESS_MONTHLY_FILE=PROC/'stage1_omb2018_historical_stress_monthly.csv'
STRESS_OUTCOMES_FILE=PROC/'stage1_omb2018_historical_stress_outcomes.csv'
HISTORIC_LONG_FILE=OUT/'stage1_omb2018_long_horizon_backtests.csv'
HISTORIC_MONTHLY_FILE=OUT/'stage1_omb2018_historical_backtest_monthly.csv'
STRESS_ORIGINS=(pd.Timestamp('2013-06-01'),pd.Timestamp('2014-06-01'),
                pd.Timestamp('2014-12-01'))
HISTORIC_HORIZON=54
score_rows, monthly_rows = [], []
for cbsa, training_row in training.iterrows():
    y = training_row.to_numpy(float)
    for origin in STRESS_ORIGINS:
        n = len(pd.date_range("2010-01-01", origin, freq="MS"))
        truth = y[n:n+HISTORIC_HORIZON]
        if len(truth) != HISTORIC_HORIZON:
            raise ValueError("Historical test window extends beyond June 2019")
        paths, _ = guarded_paths(y[:n], HISTORIC_HORIZON)
        pred = paths[FIXED_ENSEMBLE]
        for horizon in (12, 24, 36, 48, 54):
            score_rows.append({
                "cbsa_code": cbsa, "origin": origin.date().isoformat(),
                "strategy": FIXED_ENSEMBLE, "specification": FIXED_ENSEMBLE,
                "family": FIXED_ENSEMBLE, "horizon_months": horizon,
                "nmae": np.mean(np.abs(truth[:horizon] - pred[:horizon])) / np.mean(y[:n]),
                "mean_pred_minus_actual_pct": np.mean(100 * (pred[:horizon] / truth[:horizon] - 1)),
            })
        for lead, (actual, estimate) in enumerate(zip(truth, pred), start=1):
            monthly_rows.append({
                "cbsa_code": cbsa, "origin": origin.strftime("%Y_%m"), "horizon_month": lead,
                "actual_historical": actual, "forecast_historical": estimate,
                "log_actual_over_forecast": np.log(actual / estimate),
                "selected_historical_specification": FIXED_ENSEMBLE,
            })
historic_cv = pd.DataFrame(score_rows)
historical_monthly = pd.DataFrame(monthly_rows)
historic_cv.to_csv(HISTORIC_LONG_FILE, index=False)
historical_monthly.to_csv(HISTORIC_MONTHLY_FILE, index=False)
# Match the 2020-03 through 2023-12 target to historical horizons 9..54.
# The common pseudo-origin preserves some shared historical shocks across CBSAs.
historic_target=historical_monthly.loc[
    historical_monthly.horizon_month.between(9,54),
    ['cbsa_code','origin','horizon_month','log_actual_over_forecast']].copy()
historic_target['date']=pd.Timestamp('2020-03-01')+pd.to_timedelta(0,unit='D')
dates=pd.date_range('2020-03-01','2023-12-01',freq='MS')
historic_target['date']=historic_target.horizon_month.map(
    {j:date for j,date in zip(range(9,55),dates)})
observed=panel.loc[panel.cbsa_code.isin(selected.cbsa_code)&panel.date.ge('2020-03-01'),
                   ['cbsa_code','date','employment']]
current=forecast.loc[forecast.date.ge('2020-03-01'),
                     ['cbsa_code','date','counterfactual_employment']]
stress=historic_target.merge(observed,on=['cbsa_code','date'],validate='many_to_one')
stress=stress.merge(current,on=['cbsa_code','date'],validate='many_to_one')
if stress.groupby(['cbsa_code','origin']).size().ne(46).any():
    raise ValueError('Missing historical stress paths or target months')
log_cap=np.log(2.0)
stress['stress_log_ratio_capped']=stress.log_actual_over_forecast.clip(-log_cap,log_cap)
stress['stress_cap_applied']=(stress.stress_log_ratio_capped!=stress.log_actual_over_forecast)
stress['counterfactual_stress_employment']=(
    stress.counterfactual_employment*np.exp(stress.stress_log_ratio_capped))
stress['gap_pct_stress']=100*(stress.employment/stress.counterfactual_stress_employment-1)
stress['shortfall_pct_stress']=(-stress.gap_pct_stress).clip(lower=0)
stress=stress.sort_values(['cbsa_code','origin','date'])
stress.to_csv(STRESS_MONTHLY_FILE,index=False)
scenario_rows=[]
for (cbsa,origin),part in stress.groupby(['cbsa_code','origin']):
    scenario_rows.append({'cbsa_code':cbsa,'stress_origin':origin,
        'cumulative_shortfall_12_pp_months':float(part.shortfall_pct_stress.iloc[:12].sum()),
        'cumulative_shortfall_24_pp_months':float(part.shortfall_pct_stress.iloc[:24].sum()),
        'cumulative_shortfall_46_pp_months':float(part.shortfall_pct_stress.sum()),
        'signed_cumulative_gap_46_pp_months':float(-part.gap_pct_stress.sum()),
        'n_capped_months':int(part.stress_cap_applied.sum())})
stress_outcomes=pd.DataFrame(scenario_rows)
stress_outcomes.to_csv(STRESS_OUTCOMES_FILE,index=False)
print('Illustrative historical stress paths:',stress_outcomes.stress_origin.nunique(),
      '| CBSAs:',stress_outcomes.cbsa_code.nunique(),
      '| capped ratio share:',round(float(stress.stress_cap_applied.mean()),4))
print('Do NOT interpret these three historical stress paths as confidence intervals.')

primary_manifest = finish_manifest(c46_root, c46_dir, len(training), STRESS_ORIGINS,
                                   STRESS_MONTHLY_FILE, STRESS_OUTCOMES_FILE, HISTORIC_LONG_FILE)
require_primary(c46_root)
print("Published Stage 1 primary forecast:", primary_manifest["counterfactual_design"])
print("Selected publication policy:", primary_manifest["primary_policy"])
print("CBSAs:", primary_manifest["cbsa_count"], "| Forecast months:", primary_manifest["forecast_horizon_months"])
print("Per-CBSA specification selection:", primary_manifest["per_cbsa_specification_selection"])
print("Next, run Notebook 02, then Notebook 03 to recompute the resilience outcomes.")


Saved Stage 1 observed-vs-counterfactual: 43056 rows
Pre-pandemic diagnostic, July 2019--February 2020 (not used to choose models):
Region-average gap bias (pp): -0.035
Region-average absolute gap (pp): 1.468
Median region-average absolute gap (pp): 1.101
Largest diagnostic errors:
cbsa_code  mean_gap_pct  mean_absolute_pct_gap  mean_pred_minus_actual_pct
    21120    -13.907913              13.907913                   16.301380
    15700      9.653123               9.653123                   -8.658503
    39700      8.727794               8.727794                   -7.986320
    16100      8.323966               8.323966                   -7.656376
    44900     -8.168564               8.168564                    8.946664
    44780     -7.910787               7.910787                    8.659134
    31060      7.360540               7.360540                   -6.853379
    45020      6.805262               6.805262                   -6.368119
    49820     -6.547929               6.54

## Long-horizon sensitivity and forecast error

The primary cell writes forecast-horizon comparisons, restricted candidate-selection experiments,
and placebo C46 results under `output/c46_forecast_validity/`. These comparisons do not set the
primary weights. Historical stress paths are illustrative. The separate QCEW history section below
compares the training lengths and nested selection rules using the extended coverage.


In [14]:
c46_history = pd.read_csv(c46_dir / "historical_summary.csv")
display(c46_history.loc[c46_history.horizon.eq(54)])


,origin,rule,horizon,n,mean_nrmse,mean_nmae,median_placebo_c46,p90_placebo_c46,fallback_count
4,2014-06-01,damped_log_seasonal_trend,54,936,0.044902,0.039308,2.944434,171.398803,2
9,2014-06-01,fixed_ensemble,54,936,0.043777,0.038403,2.517551,163.725113,0
14,2014-06-01,log_seasonal_trend,54,936,0.047522,0.041896,15.431707,280.492207,4
19,2014-06-01,nested_selected_12,54,936,0.045193,0.039566,6.529494,216.258083,0
24,2014-06-01,nested_selected_24,54,936,0.043794,0.038325,7.580783,208.793553,0
29,2014-06-01,seasonal_drift,54,936,0.045991,0.040066,12.233683,259.860799,0
34,2014-06-01,seasonal_naive,54,936,0.049247,0.043491,0.483247,149.973973,0
39,2014-09-01,damped_log_seasonal_trend,54,936,0.044693,0.039308,3.400938,165.292464,1
44,2014-09-01,fixed_ensemble,54,936,0.043342,0.038225,3.773328,165.688127,0
49,2014-09-01,log_seasonal_trend,54,936,0.047315,0.042007,16.361091,296.369904,4


## Extended history, matched backtests and Great Recession stress test

Run the historical extension cell in Notebook 00 first. This sensitivity reuses
the guarded forecast specifications above. At the same three later forecast origins
and on the same CBSAs, compare the fixed ensemble fitted on 114 months with the
ensemble fitted on all available months since January 2001. Independently compare
12- versus 54-month selection using inner evaluation periods ending before the
outer forecast window starts. Each outer forecast is evaluated for 54 months,
with the placebo C46 measured over leads 9–54. The latest test ends in June 2019.

The three pre-recession origins use a 60-month reference because 114 months are
unavailable then. Recession stress results are reported separately. These windows
overlap and describe performance on revised QCEW histories. All comparisons within
an origin use identical CBSAs and realized outcomes. Missing months exclude that
CBSA-origin from both alternatives. The primary COVID-19 outcome continues to use
the existing 2010–2023 inputs.


In [15]:
from pathlib import Path
import os
import sys
from IPython.display import display

history_start_dir = Path(os.environ.get("RL_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
history_root = next((p for p in (history_start_dir, *history_start_dir.parents)
                     if (p / "data/processed/omb2018_geography_manifest.json").is_file()), None)
if history_root is None:
    raise FileNotFoundError("Run the Step 00 base panel cells or set RL_PROJECT_ROOT.")
if str(history_root / "code") not in sys.path:
    sys.path.insert(0, str(history_root / "code"))

from qcew_historical_validation import run_history_backtests

history_summary, history_paired = run_history_backtests(history_root)
display(history_summary)
display(history_paired)


,period,origin,rule,n_cbsa,mean_normalized_rmse_pct,mean_normalized_mae_pct,mean_signed_bias_pct,median_placebo_c46,median_signed_placebo_c46,p90_placebo_c46
0,post_recession,2012-12-01,fixed_ensemble_114m,931,5.221963,4.574543,-2.778104,0.458901,-149.038311,146.837493
1,post_recession,2012-12-01,fixed_ensemble_expanding,931,5.084190,4.420893,-2.553420,1.112925,-138.324154,145.301005
2,post_recession,2012-12-01,nested_selected_12m,931,5.408089,4.717512,-2.561276,0.887819,-141.192212,170.138734
3,post_recession,2012-12-01,nested_selected_54m,931,5.754945,5.044371,-3.257680,0.149313,-166.302833,136.063870
4,post_recession,2013-12-01,fixed_ensemble_114m,931,5.363284,4.797958,-3.060721,0.000000,-160.614358,119.933505
5,post_recession,2013-12-01,fixed_ensemble_expanding,931,5.046012,4.466103,-2.606007,0.270378,-135.587023,133.178782
6,post_recession,2013-12-01,nested_selected_12m,931,5.642160,5.005955,-3.231277,0.000000,-168.374004,135.898664
7,post_recession,2013-12-01,nested_selected_54m,931,5.826191,5.208349,-3.393609,0.000000,-177.848260,136.923484
8,post_recession,2014-12-01,fixed_ensemble_114m,931,5.280204,4.811208,-2.789608,0.000000,-153.048895,119.369698
9,post_recession,2014-12-01,fixed_ensemble_expanding,931,4.876947,4.394710,-2.176558,0.302666,-123.948319,144.994201


,period,origin,reference,alternative,matched_cbsa,mean_rmse_difference_pp,median_rmse_difference_pp,share_lower_rmse
0,post_recession,2012-12-01,fixed_ensemble_114m,fixed_ensemble_expanding,931,-0.137773,-0.084410,0.588614
1,post_recession,2012-12-01,nested_selected_12m,nested_selected_54m,931,0.346856,0.000000,0.146079
2,post_recession,2013-12-01,fixed_ensemble_114m,fixed_ensemble_expanding,931,-0.317271,-0.204348,0.665951
3,post_recession,2013-12-01,nested_selected_12m,nested_selected_54m,931,0.184031,0.000000,0.114930
4,post_recession,2014-12-01,fixed_ensemble_114m,fixed_ensemble_expanding,931,-0.403257,-0.264901,0.677766
5,post_recession,2014-12-01,nested_selected_12m,nested_selected_54m,931,0.134011,0.000000,0.169710
6,recession_stress,2006-12-01,fixed_ensemble_60m,fixed_ensemble_expanding,932,-0.274819,-0.272209,0.714592
7,recession_stress,2007-06-01,fixed_ensemble_60m,fixed_ensemble_expanding,932,-0.427540,-0.422143,0.738197
8,recession_stress,2007-09-01,fixed_ensemble_60m,fixed_ensemble_expanding,932,-0.523518,-0.503338,0.750000
